# 266건 소송 헤테로 그래프

## 설계 요약

| 노드 | 피처 | 비고 |
|------|------|------|
| `patent` | FAISS 임베딩 (title+abstract, dim=1024) | BGE-M3 |
| `company` | 소유 특허 임베딩 평균 | 소송 등장 기업만 |
| `ipc_subclass` | 없음 (relay node, learnable) | H04L 수준 |

| 엣지 | 방향 | 의미 |
|------|------|------|
| `sued` | patent(원고) → patent(피고) | **ground truth** |
| `co_litigation` | patent ↔ patent | 같은 케이스에 묶임 |
| `cites` | patent → patent | 특허 인용 (1-hop) |
| `has_ipc` | patent → ipc_subclass | IPC 분류 |
| `owns` | company → patent | 소유 관계 |
| `company_sued` | company → company | 회사 단위 소송 |

In [ ]:
# Cell 1  — 임포트 및 경로
import re, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')
plt.rcParams['figure.dpi'] = 120

ROOT  = Path('..').resolve()
SRC   = ROOT / 'src'
DATA  = ROOT / 'data' / 'samples'

# FAISS 임베딩 (소송 특허 1,331개 100% 커버 확인됨)
FAISS_INDEX = Path('/home2/jmbyeon/developments/scratch/patent_eval/uspto_db/data/embeddings/faiss.index')
PATENT_IDX  = Path('/home2/jmbyeon/developments/scratch/patent_eval/uspto_db/data/embeddings/patent_index.parquet')

print('ROOT:', ROOT)
print('FAISS:', FAISS_INDEX.exists(), '/ patent_index:', PATENT_IDX.exists())

## 1. 소송 데이터 로드

In [ ]:
# Cell 2  — 원본 로드
raw = pd.read_csv(DATA / 'both_side_patents.csv')
print(f'케이스: {raw["case_row_id"].nunique()}, rows: {len(raw)}')
raw.head(3)

In [ ]:
# Cell 3  — 특허 long 형식으로 파싱
def parse_pipe(s):
    if pd.isna(s): return []
    return [x.strip() for x in str(s).split('|') if x.strip()]

rows = []
for _, r in raw.iterrows():
    cid = r['case_row_id']
    for p in parse_pipe(r['P_patents']):
        rows.append({'case_row_id': cid, 'patent_id': p, 'side': 'plaintiff'})
    for p in parse_pipe(r['D_patents']):
        rows.append({'case_row_id': cid, 'patent_id': p, 'side': 'defendant'})

df_pats = pd.DataFrame(rows)
ALL_PATENTS = set(df_pats['patent_id'])
print(f'유니크 특허: {len(ALL_PATENTS)}, 케이스-특허 쌍: {len(df_pats)}')

## 2. 기업명 정규화

In [ ]:
# Cell 4  — 기업명 정규화 함수 + merge map 로드

_SUFFIX = re.compile(
    r',?\s*('
    r'incorporated|corporation|company|limited liability company'
    r'|limited liability partnership|limited partnership|limited'
    r'|inc|corp|co|ltd|llc|llp|lp|plc|gmbh|b\.?v|n\.?v|s\.?a|a\.?g|s\.?r\.?l'
    r'|pvt|pte|pty|ab|as|a\/s|oy|oyj|spa|srl|sarl|sas|bv|nv'
    r'|a delaware corporation|a delaware limited liability'
    r'|a california corporation|a washington limited liability'
    r'|an ohio limited liability|a texas corporation'
    r'|doing business as|terminated.*'
    r')[,.]?\s*$',
    re.IGNORECASE
)
_PREFIX = re.compile(r'^(the|a|an)\s+', re.IGNORECASE)
_JUNK   = re.compile(r'^(does |john doe|unknown|xyz |consolidated civil|consolidating civil|et al)', re.IGNORECASE)

def deep_normalize(name: str) -> str:
    s = str(name).strip()
    s = s.replace('&', 'and')
    s = re.sub(r'[^\w\s]', ' ', s)   # 구두점 → 공백
    s = re.sub(r'\s+', ' ', s).strip().upper()
    for _ in range(6):
        new = _SUFFIX.sub('', s, count=1).strip()
        if new == s: break
        s = new
    s = _PREFIX.sub('', s).strip()
    return s

# merge map 로드 (수동 검토 완료)
_merge_map_path = DATA / 'company_merge_map.csv'
MERGE_MAP = {}
if _merge_map_path.exists():
    _mm = pd.read_csv(_merge_map_path)
    MERGE_MAP = dict(zip(_mm['from'], _mm['to']))
    print(f'merge map 로드: {len(MERGE_MAP)}개')

def canonical(name: str) -> str:
    n = deep_normalize(name)
    return MERGE_MAP.get(n, n)

# 원고/피고 파싱
company_rows = []
for _, r in raw.iterrows():
    cid = r['case_row_id']
    for side, col in [('plaintiff', 'plaintiffs'), ('defendant', 'defendants')]:
        for name in parse_pipe(r[col]):
            cn = canonical(name)
            if cn and not _JUNK.match(cn) and len(cn) > 2:
                company_rows.append({'case_row_id': cid, 'raw_name': name,
                                     'norm_name': cn, 'side': side})

df_companies = pd.DataFrame(company_rows)

# 케이스별 대표 원고/피고
case_meta = []
for cid, grp in df_companies.groupby('case_row_id'):
    p_names = grp[grp['side'] == 'plaintiff']['norm_name'].tolist()
    d_names = grp[grp['side'] == 'defendant']['norm_name'].tolist()
    case_meta.append({
        'case_row_id':    cid,
        'main_plaintiff': p_names[0] if p_names else None,
        'main_defendant': d_names[0] if d_names else None,
        'all_plaintiffs': p_names,
        'all_defendants': d_names,
    })
df_cases = pd.DataFrame(case_meta)

print(f'유니크 기업(정규화+merge): {df_companies["norm_name"].nunique()}')
print(f'  원고: {df_companies[df_companies["side"]=="plaintiff"]["norm_name"].nunique()}')
print(f'  피고: {df_companies[df_companies["side"]=="defendant"]["norm_name"].nunique()}')

## 3. 외부 데이터 로드

In [ ]:
# Cell 5  — IPC (subclass 수준: H04L, G06F, ...)
ipc_raw = pd.read_csv(SRC / 'g_ipc_at_issue.tsv', sep='\t',
                      usecols=['patent_id', 'section', 'ipc_class', 'subclass'],
                      dtype=str)
ipc_raw['patent_id'] = ipc_raw['patent_id'].str.strip()
df_ipc = ipc_raw[ipc_raw['patent_id'].isin(ALL_PATENTS)].copy()

# subclass = section + ipc_class + subclass  e.g. 'H' + '04' + 'L' → 'H04L'
df_ipc['ipc_subclass'] = (
    df_ipc['section'].fillna('') +
    df_ipc['ipc_class'].fillna('') +
    df_ipc['subclass'].fillna('')
).str.strip()
df_ipc = df_ipc[df_ipc['ipc_subclass'].str.len() >= 3].drop_duplicates(['patent_id', 'ipc_subclass'])
print(f'IPC 엣지: {len(df_ipc)}, 유니크 subclass: {df_ipc["ipc_subclass"].nunique()}')

In [ ]:
# Cell 6  — Citation (소송 특허가 인용하는 모든 특허, 외부 포함)
cit_raw = pd.read_csv(SRC / 'g_us_patent_citation.tsv', sep='\t',
                      usecols=['patent_id', 'citation_patent_id'], dtype=str)
cit_raw['patent_id']          = cit_raw['patent_id'].str.strip()
cit_raw['citation_patent_id'] = cit_raw['citation_patent_id'].str.strip()

# 소송 특허가 인용하는 모든 citation (출발점만 소송 특허)
df_cit = cit_raw[cit_raw['patent_id'].isin(ALL_PATENTS)].drop_duplicates().copy()

# 내부 vs 외부 분류
internal_mask = df_cit['citation_patent_id'].isin(ALL_PATENTS)
df_cit_internal = df_cit[internal_mask]
df_cit_external = df_cit[~internal_mask]

EXTERNAL_CITED = set(df_cit_external['citation_patent_id'])

print(f'내부 citation (소송↔소송): {len(df_cit_internal)}')
print(f'외부 citation (소송→외부): {len(df_cit_external)}, 유니크 외부 특허: {len(EXTERNAL_CITED)}')

In [ ]:
# Cell 7  — Assignee: 소송 특허(patent_ownership_matched_4) + 외부 피인용 특허(g_assignee)

# ── 소송 특허 owns (이미 매칭 완료된 파일 사용)
own_raw = pd.read_csv(DATA / 'patent_ownership_matched_4.csv',
                      usecols=['case_row_id', 'patent_norm', 'final_owner'], dtype=str)
own_raw = own_raw[
    own_raw['case_row_id'].isin({str(c) for c in raw['case_row_id']}) &
    own_raw['final_owner'].notna() &
    own_raw['patent_norm'].notna()
].copy()
own_raw['org_canon'] = own_raw['final_owner'].apply(canonical)
df_asgn_lit = own_raw[own_raw['org_canon'].str.len() > 2].drop_duplicates(
    ['patent_norm', 'org_canon']
).rename(columns={'patent_norm': 'patent_id', 'org_canon': 'org_norm'})[['patent_id', 'org_norm']]

# ── 외부 피인용 특허 owns (g_assignee 사용, 소송 기업 소유인 것만)
litigation_companies = set(df_companies['norm_name'])

asgn_raw = pd.read_csv(SRC / 'g_assignee_disambiguated.tsv', sep='\t',
                       usecols=['patent_id', 'disambig_assignee_organization'], dtype=str)
asgn_raw['patent_id'] = asgn_raw['patent_id'].str.strip()
asgn_ext = asgn_raw[asgn_raw['patent_id'].isin(EXTERNAL_CITED)].dropna().copy()
asgn_ext['org_canon'] = asgn_ext['disambig_assignee_organization'].apply(canonical)
asgn_ext = asgn_ext[asgn_ext['org_canon'].isin(litigation_companies)].drop_duplicates(
    ['patent_id', 'org_canon']
).rename(columns={'org_canon': 'org_norm'})[['patent_id', 'org_norm']]

# ── 합치기
df_asgn = pd.concat([df_asgn_lit, asgn_ext], ignore_index=True).drop_duplicates(['patent_id', 'org_norm'])

# 외부 특허 중 소송 기업 소유 확인된 것
EXTERNAL_OWNED = set(asgn_ext['patent_id'])

print(f'owns 엣지 (소송 특허):  {len(df_asgn_lit)}')
print(f'owns 엣지 (외부 특허):  {len(asgn_ext)},  유니크 외부 특허: {len(EXTERNAL_OWNED)}')
print(f'owns 엣지 합계:         {len(df_asgn)}')
print(f'커버 기업: {df_asgn["org_norm"].nunique()} / {len(litigation_companies)}')

## 4. 임베딩 로드

> FAISS 인덱스: `patent_index.parquet` (6,292,658개) + `faiss.index` (dim=1024)  
> 소송 특허 1,331개 **100% 커버** 확인됨.

In [ ]:
# Cell 8  — 특허 임베딩 로드 (FAISS → vec_map)
import faiss

DIM = 1024
index    = faiss.read_index(str(FAISS_INDEX))
pid_df   = pd.read_parquet(PATENT_IDX)
pid2fidx = pid_df.set_index('patent_id')['faiss_id'].to_dict()

print(f'FAISS: {index.ntotal:,} vectors, dim={index.d}')

# 소송 특허 일괄 조회
hit_pids  = [p for p in ALL_PATENTS if p in pid2fidx]
miss_pids = [p for p in ALL_PATENTS if p not in pid2fidx]

fidxs = np.array([pid2fidx[p] for p in hit_pids], dtype='int64')
vecs  = np.empty((len(fidxs), DIM), dtype='float32')
index.reconstruct_batch(fidxs, vecs)

vec_map = {pid: vecs[i] for i, pid in enumerate(hit_pids)}

print(f'커버: {len(hit_pids)}/{len(ALL_PATENTS)} ({len(hit_pids)/len(ALL_PATENTS)*100:.1f}%)')
if miss_pids:
    print(f'미커버: {miss_pids}')

In [ ]:
# Cell 9  — Company 임베딩 = 소유 특허 임베딩 평균
company_vecs = {}
for org, grp in df_asgn.groupby('org_norm'):
    vecs = [vec_map[pid] for pid in grp['patent_id'] if pid in vec_map]
    if vecs:
        company_vecs[org] = np.mean(vecs, axis=0)

print(f'company 임베딩 생성: {len(company_vecs)} / {df_companies["norm_name"].nunique()}')

## 5. 헤테로 그래프 구성

In [ ]:
# Cell 10  — NetworkX MultiDiGraph 노드 추가
G = nx.MultiDiGraph()

side_info = df_pats.drop_duplicates('patent_id').set_index('patent_id')['side']

# patent 노드 — 소송 특허
for pid in ALL_PATENTS:
    G.add_node(f'pat_{pid}', node_type='patent', label=pid,
               side=side_info.get(pid, 'unknown'),
               is_litigation=True,
               has_emb=pid in vec_map)

# patent 노드 — 외부 피인용 특허 (소송 기업 소유 확인된 것만)
for pid in EXTERNAL_OWNED:
    G.add_node(f'pat_{pid}', node_type='patent', label=pid,
               side='external',
               is_litigation=False,
               has_emb=pid in vec_map)

# company 노드
for name in litigation_companies:
    G.add_node(f'co_{name}', node_type='company', label=name)

# ipc_subclass 노드 (relay)
for code in df_ipc['ipc_subclass'].unique():
    G.add_node(f'ipc_{code}', node_type='ipc_subclass', label=code)

type_cnt = pd.Series([d['node_type'] for _, d in G.nodes(data=True)]).value_counts()
print(f'총 노드: {G.number_of_nodes()}')
print(type_cnt)

In [ ]:
# Cell 11  — 엣지 추가

# ── sued: patent(원고) → patent(피고)  [ground truth]
for cid, grp in df_pats.groupby('case_row_id'):
    p_pats = grp[grp['side'] == 'plaintiff']['patent_id'].tolist()
    d_pats = grp[grp['side'] == 'defendant']['patent_id'].tolist()
    for pp in p_pats:
        for dp in d_pats:
            G.add_edge(f'pat_{pp}', f'pat_{dp}',
                       edge_type='sued', case_row_id=cid)

# ── co_litigation: patent ↔ patent (같은 케이스, 양방향)
for cid, grp in df_pats.groupby('case_row_id'):
    p_pats = grp[grp['side'] == 'plaintiff']['patent_id'].tolist()
    d_pats = grp[grp['side'] == 'defendant']['patent_id'].tolist()
    for pp in p_pats:
        for dp in d_pats:
            G.add_edge(f'pat_{pp}', f'pat_{dp}',
                       edge_type='co_litigation', case_row_id=cid)
            G.add_edge(f'pat_{dp}', f'pat_{pp}',
                       edge_type='co_litigation', case_row_id=cid)

# ── cites: 소송 특허 → 피인용 특허 (내부 + 외부, 단 외부는 EXTERNAL_OWNED만)
for _, r in df_cit.iterrows():
    tgt = r['citation_patent_id']
    if tgt in ALL_PATENTS or tgt in EXTERNAL_OWNED:
        G.add_edge(f'pat_{r["patent_id"]}', f'pat_{tgt}', edge_type='cites')

# ── has_ipc: patent → ipc_subclass
for _, r in df_ipc.iterrows():
    G.add_edge(f'pat_{r["patent_id"]}', f'ipc_{r["ipc_subclass"]}',
               edge_type='has_ipc')

# ── owns: company → patent (소송 특허 + 외부 특허)
for _, r in df_asgn.iterrows():
    if f'pat_{r["patent_id"]}' in G:
        G.add_edge(f'co_{r["org_norm"]}', f'pat_{r["patent_id"]}',
                   edge_type='owns')

# ── company_sued: company → company
for _, r in df_cases.iterrows():
    if r['main_plaintiff'] and r['main_defendant']:
        G.add_edge(f'co_{r["main_plaintiff"]}', f'co_{r["main_defendant"]}',
                   edge_type='company_sued', case_row_id=r['case_row_id'])

et_cnt = pd.Series([d['edge_type'] for _, _, d in G.edges(data=True)]).value_counts()
print(f'총 엣지: {G.number_of_edges()}')
print(et_cnt)

## 6. 기본 통계

In [ ]:
# Cell 12  — 케이스별 특허 수 분포
pat_per_case = df_pats.groupby('case_row_id')['patent_id'].count()
p_per_case   = df_pats[df_pats['side']=='plaintiff'].groupby('case_row_id')['patent_id'].count()
d_per_case   = df_pats[df_pats['side']=='defendant'].groupby('case_row_id')['patent_id'].count()

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, data, title in zip(axes,
    [pat_per_case, p_per_case, d_per_case],
    ['전체 특허 / 케이스', '원고 특허 / 케이스', '피고 특허 / 케이스']):
    ax.hist(data, bins=20, color='steelblue', edgecolor='white')
    ax.axvline(data.median(), color='red', linestyle='--', label=f'median={data.median():.0f}')
    ax.set_title(title); ax.set_xlabel('특허 수'); ax.legend()
plt.tight_layout(); plt.show()

In [ ]:
# Cell 13  — 소송 최다 기업 Top 20
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
for ax, side, color in zip(axes, ['plaintiff', 'defendant'], ['#2196F3', '#F44336']):
    top = df_companies[df_companies['side']==side]['norm_name'].value_counts().head(20)
    ax.barh(top.index[::-1], top.values[::-1], color=color, alpha=0.85)
    ax.set_title(f'{side} Top 20'); ax.set_xlabel('케이스 수')
plt.tight_layout(); plt.show()

## 7. IPC 분석

In [ ]:
# Cell 14  — IPC section별 원고/피고 분포
IPC_LABELS = {'A':'Human Necessities','B':'Operations','C':'Chemistry',
              'D':'Textiles','E':'Constructions','F':'Mechanical Eng',
              'G':'Physics','H':'Electricity'}

df_pats_ipc = df_pats.merge(
    df_ipc[['patent_id','ipc_subclass']].assign(section=df_ipc['section']).drop_duplicates(),
    on='patent_id', how='left'
)
sec_side = df_pats_ipc.dropna(subset=['section']).groupby(['section','side']).size().unstack(fill_value=0)
sec_side.index = [f"{s}: {IPC_LABELS.get(s,s)}" for s in sec_side.index]

sec_side.plot(kind='bar', figsize=(12,5), color=['#2196F3','#F44336'], alpha=0.85)
plt.title('IPC Section — 원고 vs 피고')
plt.xticks(rotation=25, ha='right'); plt.tight_layout(); plt.show()

In [ ]:
# Cell 15  — 케이스별 IPC Jaccard 유사도 (원고 ↔ 피고)
overlap_rows = []
for cid, grp in df_pats_ipc.groupby('case_row_id'):
    p_ipc = set(grp[grp['side']=='plaintiff']['ipc_subclass'].dropna())
    d_ipc = set(grp[grp['side']=='defendant']['ipc_subclass'].dropna())
    if p_ipc and d_ipc:
        inter, union = len(p_ipc & d_ipc), len(p_ipc | d_ipc)
        overlap_rows.append({'case_row_id': cid,
                             'jaccard': inter/union if union else 0,
                             'overlap_n': inter})
df_ov = pd.DataFrame(overlap_rows)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].hist(df_ov['jaccard'], bins=25, color='teal', edgecolor='white')
axes[0].axvline(df_ov['jaccard'].median(), color='red', linestyle='--',
                label=f"median={df_ov['jaccard'].median():.3f}")
axes[0].set_title('IPC Jaccard 유사도 (원고↔피고)'); axes[0].legend()
axes[1].hist(df_ov['overlap_n'], bins=20, color='darkorange', edgecolor='white')
axes[1].set_title('겹치는 IPC subclass 수')
plt.tight_layout(); plt.show()
print(df_ov[['jaccard','overlap_n']].describe().round(3))

## 8. Citation 분석

In [ ]:
# Cell 16  — 소송 쌍 간 직접 인용 여부
G_cit = nx.DiGraph()
G_cit.add_nodes_from(ALL_PATENTS)
for _, r in df_cit.iterrows():
    G_cit.add_edge(r['patent_id'], r['citation_patent_id'])

cite_rows = []
for cid, grp in df_pats.groupby('case_row_id'):
    p_list = grp[grp['side']=='plaintiff']['patent_id'].tolist()
    d_list = grp[grp['side']=='defendant']['patent_id'].tolist()
    for pp in p_list:
        for dp in d_list:
            cite_rows.append({
                'case_row_id': cid,
                'p_cites_d': G_cit.has_edge(pp, dp),
                'd_cites_p': G_cit.has_edge(dp, pp),
            })
df_cp = pd.DataFrame(cite_rows)
df_cp['any'] = df_cp['p_cites_d'] | df_cp['d_cites_p']

n_with = df_cp[df_cp['any']]['case_row_id'].nunique()
total  = df_cp['case_row_id'].nunique()
print(f'직접 인용 관계 있는 케이스: {n_with}/{total} ({n_with/total*100:.1f}%)')
print(f'  원고→피고: {df_cp["p_cites_d"].sum()}')
print(f'  피고→원고: {df_cp["d_cites_p"].sum()}')

In [ ]:
# Cell 17  — 인용 degree 원고/피고 비교
in_deg  = dict(G_cit.in_degree())
out_deg = dict(G_cit.out_degree())
df_pats['in_deg']  = df_pats['patent_id'].map(in_deg).fillna(0).astype(int)
df_pats['out_deg'] = df_pats['patent_id'].map(out_deg).fillna(0).astype(int)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, col, title in zip(axes, ['in_deg','out_deg'], ['피인용수 (in)','인용수 (out)']):
    for side, color in [('plaintiff','#2196F3'),('defendant','#F44336')]:
        ax.hist(df_pats[df_pats['side']==side][col].clip(upper=40),
                bins=20, alpha=0.6, color=color, label=side)
    ax.set_title(title); ax.legend()
plt.tight_layout(); plt.show()
print(df_pats.groupby('side')[['in_deg','out_deg']].describe().round(2))

## 9. 중심성 & 커뮤니티 분석

In [ ]:
# Cell 18  — co_litigation 그래프 중심성
G_co = nx.Graph()
for u, v, d in G.edges(data=True):
    if d['edge_type'] == 'co_litigation':
        G_co.add_edge(u.replace('pat_',''), v.replace('pat_',''))

deg_c = nx.degree_centrality(G_co)
bet_c = nx.betweenness_centrality(G_co)

df_cent = pd.DataFrame({
    'patent_id': list(deg_c),
    'degree_c':  list(deg_c.values()),
    'between_c': [bet_c[p] for p in deg_c],
}).sort_values('between_c', ascending=False)
df_cent['side'] = df_cent['patent_id'].map(
    df_pats.drop_duplicates('patent_id').set_index('patent_id')['side'])

fig, ax = plt.subplots(figsize=(9,6))
for side, color in [('plaintiff','#2196F3'),('defendant','#F44336')]:
    sub = df_cent[df_cent['side']==side]
    ax.scatter(sub['degree_c'], sub['between_c'], c=color, alpha=0.6, s=35, label=side)
for _, row in df_cent.head(8).iterrows():
    ax.annotate(row['patent_id'], (row['degree_c'], row['between_c']), fontsize=7)
ax.set_xlabel('Degree Centrality'); ax.set_ylabel('Betweenness')
ax.set_title('특허 중심성 (co-litigation 그래프)'); ax.legend()
plt.tight_layout(); plt.show()
print(df_cent.head(10)[['patent_id','side','degree_c','between_c']].to_string(index=False))

In [ ]:
# Cell 19  — 커뮤니티 탐지
try:
    from community import best_partition
    partition = best_partition(G_co, random_state=42)
    method = 'Louvain'
except ImportError:
    comms = nx.community.greedy_modularity_communities(G_co)
    partition = {n: i for i, c in enumerate(comms) for n in c}
    method = 'Greedy Modularity'

df_cent['community'] = df_cent['patent_id'].map(partition)
print(f'[{method}] 커뮤니티 수: {df_cent["community"].nunique()}')
print(df_cent['community'].value_counts().head(10))

In [ ]:
# Cell 20  — 커뮤니티별 IPC 분포
top_comms = df_cent['community'].value_counts().head(5).index.tolist()
rows = []
for comm in top_comms:
    pats = df_cent[df_cent['community']==comm]['patent_id'].tolist()
    for sec in df_ipc[df_ipc['patent_id'].isin(pats)]['section'].dropna():
        rows.append({'community': comm, 'section': sec})

if rows:
    pivot = pd.DataFrame(rows).groupby(['community','section']).size().unstack(fill_value=0)
    pivot.plot(kind='bar', figsize=(12,5), colormap='tab10', alpha=0.85)
    plt.title('커뮤니티별 IPC Section (Top 5 커뮤니티)')
    plt.xticks(rotation=0); plt.tight_layout(); plt.show()

## 10. Graphistry 시각화

> Free tier: `mode='public'` 필수. `g.plot()` → `hub.graphistry.com` URL 반환.

In [ ]:
# Cell 21  — Graphistry 등록
try:
    import graphistry
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'graphistry', '-q'])
    import graphistry

graphistry.register(
    api=3, protocol='https', server='hub.graphistry.com',
    username='jungmin',
    password='wjdalsvnf3!',
)
graphistry.privacy(mode='public')
print('graphistry', graphistry.__version__)

In [ ]:
# Cell 22  — 노드/엣지 DataFrame 준비
NODE_COLOR = {
    'patent_plaintiff': '#4FC3F7',
    'patent_defendant': '#EF9A9A',
    'company':          '#A5D6A7',
    'ipc_subclass':     '#FFE082',
}
NODE_SIZE = {'patent': 8, 'company': 18, 'ipc_subclass': 6}
EDGE_COLOR = {
    'sued':          '#EF5350',
    'co_litigation': '#AB47BC',
    'cites':         '#29B6F6',
    'has_ipc':       '#FFD54F',
    'owns':          '#66BB6A',
    'company_sued':  '#FF7043',
}

node_rows = []
for nid, d in G.nodes(data=True):
    nt = d['node_type']
    lbl = d.get('label', nid)
    if nt == 'patent':
        color = NODE_COLOR.get(f'patent_{d.get("side","")}', '#aaaaaa')
    else:
        color = NODE_COLOR.get(nt, '#cccccc')
    node_rows.append({'id': nid, 'label': str(lbl)[:30],
                      'node_type': nt, 'color': color,
                      'size': NODE_SIZE.get(nt, 8)})
df_gn = pd.DataFrame(node_rows)

edge_rows = []
for u, v, d in G.edges(data=True):
    et = d['edge_type']
    edge_rows.append({'src': u, 'dst': v, 'edge_type': et,
                      'color': EDGE_COLOR.get(et, '#888888')})
df_ge = pd.DataFrame(edge_rows)
print(f'노드: {len(df_gn)}, 엣지: {len(df_ge)}')
print(df_ge['edge_type'].value_counts())

In [ ]:
# Cell 23  — 뷰 1: sued + co_litigation + cites + owns
SHOW = {'sued', 'co_litigation', 'cites', 'owns'}
df_e = df_ge[df_ge['edge_type'].isin(SHOW)]
df_n = df_gn[df_gn['id'].isin(set(df_e['src']) | set(df_e['dst']))]

g1 = (graphistry
      .nodes(df_n, 'id').edges(df_e, 'src', 'dst')
      .bind(node_label='label', node_color='color', node_size='size',
            edge_color='color', edge_title='edge_type')
      .settings(url_params={'play': 3000, 'strongGravity': True}))
print('URL:', g1.plot(render=False))
g1.plot()

In [ ]:
# Cell 24  — 뷰 2: 회사 소송 네트워크 (company_sued)
df_e2 = df_ge[df_ge['edge_type'] == 'company_sued']
df_n2 = df_gn[df_gn['id'].isin(set(df_e2['src']) | set(df_e2['dst']))].copy()

# 소송 건수 → 노드 크기
cnt = pd.concat([df_e2['src'], df_e2['dst']]).value_counts()
df_n2['size'] = (df_n2['id'].map(cnt).fillna(1) * 3 + 10).clip(upper=60)

g2 = (graphistry
      .nodes(df_n2, 'id').edges(df_e2, 'src', 'dst')
      .bind(node_label='label', node_color='color', node_size='size',
            edge_color='color', edge_title='edge_type')
      .settings(url_params={'play': 5000}))
print('URL:', g2.plot(render=False))
g2.plot()

In [ ]:
# Cell 25  — 뷰 3: 특정 케이스 서브그래프
TARGET_CASE = raw['case_row_id'].iloc[0]  # 원하는 case_row_id로 변경

focus = {f'case_{TARGET_CASE}'}
focus |= {f'pat_{p}' for p in df_pats[df_pats['case_row_id']==TARGET_CASE]['patent_id']}

df_e3 = df_ge[df_ge['src'].isin(focus) | df_ge['dst'].isin(focus)]
df_n3 = df_gn[df_gn['id'].isin(set(df_e3['src']) | set(df_e3['dst']))]

g3 = (graphistry
      .nodes(df_n3, 'id').edges(df_e3, 'src', 'dst')
      .bind(node_label='label', node_color='color', node_size='size',
            edge_color='color', edge_title='edge_type'))
print(f'Case {TARGET_CASE}: 노드 {len(df_n3)}, 엣지 {len(df_e3)}')
g3.plot()

## 11. 그래프 요약 & 저장

In [ ]:
# Cell 26  — 전체 요약
print('=' * 55)
print('헤테로 그래프 요약')
print('=' * 55)
print('[노드]')
for nt, cnt in pd.Series([d['node_type'] for _,d in G.nodes(data=True)]).value_counts().items():
    print(f'  {nt:<15} {cnt:>6}')
print(f'  {"합계":<15} {G.number_of_nodes():>6}')
print('[엣지]')
for et, cnt in pd.Series([d['edge_type'] for _,_,d in G.edges(data=True)]).value_counts().items():
    print(f'  {et:<15} {cnt:>6}')
print(f'  {"합계":<15} {G.number_of_edges():>6}')
print()
print(f'임베딩 커버: {len(vec_map)}/{len(ALL_PATENTS)} 특허')
print(f'company 임베딩: {len(company_vecs)}개')

In [ ]:
# Cell 27  — GraphML 저장 (Gephi 호환)
G_s = nx.DiGraph()
for n, d in G.nodes(data=True):
    G_s.add_node(n, **{k: str(v) for k, v in d.items()})
for u, v, d in G.edges(data=True):
    if not G_s.has_edge(u, v):
        G_s.add_edge(u, v, **{k: str(v2) for k, v2 in d.items()})

out = SRC / 'graph_266.graphml'
nx.write_graphml(G_s, str(out))
print(f'저장: {out}')